# 03 — Validate the joins locally

Runs the same joining SQL the production pipelines use, so you can sanity-check
the dataset *before* deploying. If these queries return zero rows or skewed
distributions, the sync will faithfully replicate that into Cosmos.


In [ ]:
# Load .env from the repo root so we use the same settings as the sync job.
from pathlib import Path
import os

from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / ".env.example").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not find repo root (.env.example missing)")
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

BQ_PROJECT = os.environ["BQ_PROJECT_ID"]
BQ_DATASET = os.environ.get("BQ_DATASET", "learnsphere")
BQ_LOCATION = os.environ.get("BQ_LOCATION", "US")
print(f"project={BQ_PROJECT}  dataset={BQ_DATASET}  location={BQ_LOCATION}")


In [ ]:
from google.cloud import bigquery
import pandas as pd

bq = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

def q(sql: str) -> pd.DataFrame:
    return bq.query(sql).result().to_dataframe(create_bqstorage_client=False)


## Courses ⨝ instructors ⨝ agg(reviews)


In [ ]:
sql = f"""
WITH review_stats AS (
  SELECT course_id, COUNT(*) AS review_count, AVG(rating) AS avg_rating
  FROM `{BQ_PROJECT}.{BQ_DATASET}.course_reviews`
  GROUP BY course_id
)
SELECT c.course_id, c.title, c.category, i.display_name AS instructor,
       COALESCE(rs.review_count, 0) AS reviews, ROUND(rs.avg_rating, 2) AS avg_rating
FROM `{BQ_PROJECT}.{BQ_DATASET}.courses` c
LEFT JOIN `{BQ_PROJECT}.{BQ_DATASET}.instructors` i USING (instructor_id)
LEFT JOIN review_stats rs USING (course_id)
ORDER BY reviews DESC
LIMIT 10
"""
q(sql)


## Learners — effective watermark distribution


In [ ]:
sql = f"""
WITH learner_activity AS (
  SELECT learner_id, MAX(last_activity_at) AS max_activity
  FROM `{BQ_PROJECT}.{BQ_DATASET}.enrollments`
  GROUP BY learner_id
)
SELECT
  TIMESTAMP_TRUNC(GREATEST(l.updated_at, COALESCE(la.max_activity, l.updated_at)), DAY) AS day,
  COUNT(*) AS learners
FROM `{BQ_PROJECT}.{BQ_DATASET}.learners` l
LEFT JOIN learner_activity la USING (learner_id)
GROUP BY day
ORDER BY day DESC
LIMIT 14
"""
q(sql)


## Recommendations sanity check

Top-3 candidate courses for a few sample learners. Empty results usually mean
the learner has no enrollments yet — fine.


In [ ]:
sql = f"""
WITH learner_category_minutes AS (
  SELECT e.learner_id, c.category,
         SUM(c.duration_minutes * SAFE_DIVIDE(e.progress_percent, 100.0)) AS minutes
  FROM `{BQ_PROJECT}.{BQ_DATASET}.enrollments` e
  JOIN `{BQ_PROJECT}.{BQ_DATASET}.courses` c USING (course_id)
  GROUP BY e.learner_id, c.category
),
preferred AS (
  SELECT learner_id, category FROM (
    SELECT learner_id, category,
           ROW_NUMBER() OVER (PARTITION BY learner_id ORDER BY minutes DESC) AS rk
    FROM learner_category_minutes
  ) WHERE rk = 1
),
popularity AS (
  SELECT course_id, COUNT(*) AS enroll_count
  FROM `{BQ_PROJECT}.{BQ_DATASET}.enrollments`
  GROUP BY course_id
)
SELECT p.learner_id, p.category, c.course_id, c.title, COALESCE(pop.enroll_count, 0) AS popularity
FROM preferred p
JOIN `{BQ_PROJECT}.{BQ_DATASET}.courses` c USING (category)
LEFT JOIN popularity pop USING (course_id)
WHERE p.learner_id IN (
  SELECT learner_id FROM `{BQ_PROJECT}.{BQ_DATASET}.learners` LIMIT 3
)
ORDER BY p.learner_id, popularity DESC
LIMIT 12
"""
q(sql)
